In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

2025-12-08 14:32:49.452464: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 14:32:49.456149: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
from pathlib import Path

In [3]:
DATA_ROOT=Path("/home/mcn26/project_pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251206"

In [4]:
spread_hypothesis_20251119=scm.HypothesisSet.from_tsv(path/"spread_hypothesis_20251119.tsv")

In [5]:
spread_hypothesis_20251119.df

,comparison_CRE,reference_CRE,comparison_cell_type,reference_cell_type,meta
0,inactive_0,reference,Cardiomyocytes,Cardiomyocytes,<NA>
1,inactive_0,reference,EpiblastPrimitiveStreak,EpiblastPrimitiveStreak,<NA>
2,inactive_0,reference,ExEndodermParietal,ExEndodermParietal,<NA>
3,inactive_0,reference,ExEndodermVisceral,ExEndodermVisceral,<NA>
4,inactive_0,reference,Haematoendothelial,Haematoendothelial,<NA>
...,...,...,...,...,...
5495,active_45,active_45,reference,SurfaceEctoderm,<NA>
5496,active_46,active_46,reference,SurfaceEctoderm,<NA>
5497,active_47,active_47,reference,SurfaceEctoderm,<NA>
5498,active_48,active_48,reference,SurfaceEctoderm,<NA>


In [9]:
from dask.distributed import get_client#Semaphore, as_completed,

In [10]:
ortho_root=path/name/"orthos_with_precomputed_wald"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=5, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    #sem = Semaphore(name="test")
    #with sem:
    client=get_client()
    ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
    scmpradat_oi=scm.scMPRA_data.from_parquet(path/"sim_with_orthos_20251206"/"scMPRA"/f"{name}.scmpra")
    ortho_oi.training_data=scmpradat_oi
    print(dir(ortho_oi))
    runner = scm.HypothesisTester(test_type)
    output_short=Path(output_root)/hypothesis_set_name/test_type
    output_short.mkdir(exist_ok=True,parents=True)
    runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)
    #runner.run(hypothesis_set, scmpradat_oi, client).to_tsv(output_short/name)

In [11]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=4,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=1)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [ ]:
#futures_wald = [client.submit(compute_one_wald,
#                        input_root=ortho_root,
#                        name=name_oi,
#                        output_root=output_root,
#                        hypothesis_set=spread_hypothesis_20251119,
#                        hypothesis_set_name="spread_hypothesis_20251119",
#                        test_type="wald") for name_oi in input_ortho_names]

In [12]:
futures_mwu = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [13]:
client.dashboard_link

'http://127.0.0.1:36773/status'

In [ ]:
futures_mwu[0].result()

In [ ]:
client.close()
cluster.close()